In [1]:
import torch
from train import run_baseline_experiment, run_lora_experiment
from utils import plot_results
#print("CUDA available?", torch.cuda.is_available())

import os
import subprocess

myrepr = lambda x: repr(round(x, 8)).replace('.',',') if isinstance(x, float) else repr(x)


Seed set to 42


### Bernoulli-LoRA (tunning)

In [2]:
%%writefile run_single_lora_experiment.py
#!/usr/bin/env python3

import argparse
from train import run_lora_experiment, run_lora_experiment_multiple_seeds
import numpy as np
import os

myrepr = lambda x: repr(round(x, 8)).replace('.',',') if isinstance(x, float) else repr(x)

def str2bool(v):
    return v.lower() in ("true", "1", "yes")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--merge_frequency", type=int, default=1)
    parser.add_argument("--prob", type=float, default=0.5)
    parser.add_argument("--init_train_zero", type=str2bool, default=False)
    parser.add_argument("--gaussian_resample", type=str2bool, default=False)
    parser.add_argument("--deterministic_init", type=str2bool, default=False)
    parser.add_argument("--train_A", type=str2bool, default=True)
    parser.add_argument("--train_B", type=str2bool, default=True)
    parser.add_argument("--init_method_A", type=str, default="zero")
    parser.add_argument("--init_method_B", type=str, default="gaussian")
    parser.add_argument("--use_stochastic", type=str2bool, default=True)

    args = parser.parse_args()

    # Create a job name
    job_name_parts = [
        f"mf-{args.merge_frequency}",
        f"p-{myrepr(args.prob)}",
        f"itZ-{int(args.init_train_zero)}",
        f"gR-{int(args.gaussian_resample)}",
        f"det-{int(args.deterministic_init)}",
        f"stoch-{int(args.use_stochastic)}",
        f"trainA-{int(args.train_A)}",
        f"trainB-{int(args.train_B)}",
        f"initA-{args.init_method_A}",
        f"initB-{args.init_method_B}"
    ]
    job_name = "_".join(job_name_parts)

    NUM_LAUNCHES = 20
    seeds_to_try = np.arange(1, NUM_LAUNCHES + 1)
    
    acc, median_acc, mean_acc, std_acc = run_lora_experiment_multiple_seeds(
        seeds=seeds_to_try,
        rank=1,                               
        train_A=args.train_A,
        train_B=args.train_B,
        init_method_A=args.init_method_A,
        init_method_B=args.init_method_B,
        merge_frequency=args.merge_frequency,
        use_stochastic=args.use_stochastic,                  
        prob=args.prob,
        deterministic_init=args.deterministic_init,
        init_train_zero=args.init_train_zero,
        gaussian_resample=args.gaussian_resample
    )

    output_dir = "slurm_outs"
    os.makedirs(output_dir, exist_ok=True)

    np.save(f"{output_dir}/acc_{job_name}.npy", acc)
    np.save(f"{output_dir}/std-acc_{job_name}.npy", std_acc)
    np.save(f"{output_dir}/median-acc_{job_name}.npy", median_acc)
    np.save(f"{output_dir}/mean-acc_{job_name}.npy", mean_acc)
    

Overwriting run_single_lora_experiment.py


In [3]:
def submit_slurm_job(
    job_name,
    script_args,
    output_dir="slurm_outs",
    sh_folder="slurm_sh",
    partition="batch",
    time="4:00:00",
    mem="8GB",
    gpus=1,
    cpus=16,
    submit=True
):
    """
    Creates a Slurm shell script and optionally submits it using `sbatch`.
    
    Parameters
    ----------
    job_name : str
        Name of the job used in Slurm (#SBATCH --job-name).
    script_args : list of str
        The arguments to pass to the Python script.
    output_dir : str
        Directory for storing .out logs.
    sh_folder : str
        Directory for storing .sh scripts.
    partition : str
        Partition name for Slurm.
    time : str
        Time limit for the job (e.g. '2:00:00').
    mem : str
        Memory limit (e.g. '8GB').
    gpus : int
        Number of GPUs to request.
    cpus : int
        Number of CPU cores to request.
    submit : bool
        If True, call `sbatch` immediately. If False, just write the script and do not submit.
    """
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(sh_folder, exist_ok=True)
    
    sh_name = os.path.join(sh_folder, f"{job_name}.sh")
    with open(sh_name, "w") as f:
        f.write("#!/bin/bash\n")
        f.write(f"#SBATCH --job-name={job_name}\n")
        f.write(f"#SBATCH --cpus-per-task={cpus}\n")
        f.write(f"#SBATCH --gpus={gpus}\n")
        f.write("#SBATCH -N 1\n")
        f.write(f"#SBATCH --partition={partition}\n")
        f.write(f"#SBATCH --mem={mem}\n")
        f.write(f"#SBATCH --time={time}\n")
        f.write(f"#SBATCH --output={output_dir}/{job_name}.out\n")

        f.write("\n")
        f.write("python run_single_lora_experiment.py")
        for arg in script_args:
            f.write(f" {arg}")
        f.write("\n")

    if submit:
        result = subprocess.run(["sbatch", sh_name], capture_output=True, text=True)
        if result.returncode == 0:
            print(f"[SUBMITTED] Job {job_name} => {result.stdout.strip()}")
        else:
            print(f"[ERROR] sbatch failed for {job_name}: {result.stderr.strip()}")

In [ ]:

# LoRA (A:Gaussian; B:Zero) and LoRA (A:Zero; B:Gaussian)
# merge_freq_list, prob_list = [0], [0.5]
# init_train_zero_list,gaussian_resample_list = [True], [True]
# deterministic_init, use_stochastic = True, False
# param_combos = [dict(train_A=True, train_B=True, init_method_A="zero", init_method_B="gaussian"),
#                 dict(train_A=True, train_B=True, init_method_A="gaussian", init_method_B="zero")]
#----------------------------------------------
# CoLA (A:Gaussian; B:Zero) and CoLA (A:Zero; B:Gaussian)
# merge_freq_list, prob_list = [1], [0.5]
# init_train_zero_list,gaussian_resample_list = [True], [True]
# deterministic_init, use_stochastic = True, False
# param_combos = [dict(train_A=True, train_B=True, init_method_A="zero", init_method_B="gaussian"),
#                 dict(train_A=True, train_B=True, init_method_A="gaussian", init_method_B="zero")]
#----------------------------------------------
# AsymmLoRA (A:Gaussian; B:Zero) and AsymmLoRA (A:Zero; B:Gaussian)
# merge_freq_list, prob_list = [0], [0.5]
# init_train_zero_list,gaussian_resample_list = [True], [True]
# deterministic_init, use_stochastic = True, False
# param_combos = [dict(train_A=True, train_B=False, init_method_A="zero", init_method_B="gaussian"),
#                 dict(train_A=False, train_B=True, init_method_A="gaussian", init_method_B="zero")]

# RAC-LoRA (A:Gaussian; B:Zero) and RAC-LoRA (A:Zero; B:Gaussian)
# merge_freq_list, prob_list = [1], [0.5]
# init_train_zero_list, gaussian_resample_list = [True], [True]
# deterministic_init, use_stochastic = True, False
# param_combos = [dict(train_A=True, train_B=False, init_method_A="zero", init_method_B="gaussian"),
#                 dict(train_A=False, train_B=True, init_method_A="gaussian", init_method_B="zero")]

# Bernoulli-LoRA (A:Gaussian; B:Zero) and Bernoulli-LoRA (A:Zero; B:Gaussian)
merge_freq_list, prob_list = [1], [0.01, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.99]
init_train_zero_list, gaussian_resample_list = [True, False], [True, False]
# deterministic_init, use_stochastic = True, True
# param_combos = [dict(train_A=True, train_B=False, init_method_A="zero", init_method_B="gaussian"),
#                 dict(train_A=False, train_B=True, init_method_A="gaussian", init_method_B="zero")]

deterministic_init, use_stochastic = False, True
param_combos = [dict(train_A=True, train_B=False, init_method_A="zero", init_method_B="gaussian")]




if __name__ == "__main__":
    job_idx = 0
    
    for merge_freq in merge_freq_list:
        for prob in prob_list:
            for init_train_zero in init_train_zero_list:
                for gaussian_resample in gaussian_resample_list:
                    for combo in param_combos:
                        job_name_parts = [
                            f"mf-{merge_freq}",
                            f"p-{myrepr(prob)}",
                            f"itZ-{int(init_train_zero)}",
                            f"gR-{int(gaussian_resample)}",
                            f"det-{int(deterministic_init)}",
                            f"stoch-{int(use_stochastic)}",
                            f"trainA-{int(combo['train_A'])}",
                            f"trainB-{int(combo['train_B'])}",
                            f"initA-{combo['init_method_A']}",
                            f"initB-{combo['init_method_B']}"
                        ]
                        job_name = "_".join(str(x) for x in job_name_parts)
                        
                        script_args = [
                            f"--merge_frequency={merge_freq}",
                            f"--prob={prob}",
                            f"--init_train_zero={init_train_zero}",
                            f"--gaussian_resample={gaussian_resample}",
                            f"--deterministic_init={deterministic_init}",
                            f"--train_A={combo['train_A']}",
                            f"--train_B={combo['train_B']}",
                            f"--init_method_A={combo['init_method_A']}",
                            f"--init_method_B={combo['init_method_B']}",
                            f"--use_stochastic={use_stochastic}"    
                        ]
                        
                        print(f"[{job_idx}] Creating job: {job_name}")
                        submit_slurm_job(
                            job_name=job_name,
                            script_args=script_args,
                            output_dir="slurm_outs",
                            sh_folder="slurm_sh",
                            partition="batch",
                            time="6:00:00",
                            mem="12GB",
                            gpus=1,
                            cpus=16,
                            submit=0  # set to True to actually submit
                        )
                        job_idx += 1

    print(f"Done. Created {job_idx} job scripts.")


### Relaunch problematic experiments

In [4]:
def parse_job_name(job_name):
    """
    Parse a job name like:
        mf-1_p-0,4_itZ-1_gR-1_det-1_stoch-1_trainA-1_trainB-0_initA-zero_initB-gaussian
    into a dictionary that matches the arguments of run_single_lora_experiment.py
    """
    parts = job_name.split('_')
    d = {}
    for part in parts:
        # e.g. part = 'mf-1', 'p-0,4', 'itZ-1', etc.
        if part.startswith('mf-'):
            d['merge_frequency'] = int(part.split('-', 1)[1])
        elif part.startswith('p-'):
            # convert 0,4 --> 0.4
            val_str = part.split('-', 1)[1].replace(',', '.')
            d['prob'] = float(val_str)
        elif part.startswith('itZ-'):
            val = int(part.split('-', 1)[1])
            d['init_train_zero'] = bool(val)
        elif part.startswith('gR-'):
            val = int(part.split('-', 1)[1])
            d['gaussian_resample'] = bool(val)
        elif part.startswith('det-'):
            val = int(part.split('-', 1)[1])
            d['deterministic_init'] = bool(val)
        elif part.startswith('stoch-'):
            val = int(part.split('-', 1)[1])
            d['use_stochastic'] = bool(val)
        elif part.startswith('trainA-'):
            val = int(part.split('-', 1)[1])
            d['train_A'] = bool(val)
        elif part.startswith('trainB-'):
            val = int(part.split('-', 1)[1])
            d['train_B'] = bool(val)
        elif part.startswith('initA-'):
            d['init_method_A'] = part.split('-', 1)[1]
        elif part.startswith('initB-'):
            d['init_method_B'] = part.split('-', 1)[1]
        else:
            print(f"Warning: unrecognized part {part}")
    return d


if __name__ == "__main__":
    # List all problematic jobs from your "Data NOT found for job: ..." lines
    problematic_jobs = [
        "mf-1_p-0,4_itZ-1_gR-1_det-1_stoch-1_trainA-1_trainB-0_initA-zero_initB-gaussian",
        "mf-1_p-0,5_itZ-1_gR-1_det-1_stoch-1_trainA-1_trainB-0_initA-zero_initB-gaussian",
        "mf-1_p-0,4_itZ-1_gR-0_det-1_stoch-1_trainA-1_trainB-0_initA-zero_initB-gaussian",
        "mf-1_p-0,4_itZ-0_gR-1_det-1_stoch-1_trainA-1_trainB-0_initA-zero_initB-gaussian",
        "mf-1_p-0,6_itZ-0_gR-1_det-1_stoch-1_trainA-1_trainB-0_initA-zero_initB-gaussian",
        "mf-1_p-0,4_itZ-1_gR-1_det-1_stoch-1_trainA-0_trainB-1_initA-gaussian_initB-zero",
        "mf-1_p-0,5_itZ-1_gR-1_det-1_stoch-1_trainA-0_trainB-1_initA-gaussian_initB-zero",
        "mf-1_p-0,4_itZ-1_gR-0_det-1_stoch-1_trainA-0_trainB-1_initA-gaussian_initB-zero",
        "mf-1_p-0,4_itZ-0_gR-1_det-1_stoch-1_trainA-0_trainB-1_initA-gaussian_initB-zero",
        "mf-1_p-0,5_itZ-0_gR-1_det-1_stoch-1_trainA-0_trainB-1_initA-gaussian_initB-zero",
        "mf-1_p-0,3_itZ-0_gR-0_det-1_stoch-1_trainA-0_trainB-1_initA-gaussian_initB-zero",
        "mf-1_p-0,4_itZ-0_gR-0_det-1_stoch-1_trainA-0_trainB-1_initA-gaussian_initB-zero"
    ]

    for job_name in problematic_jobs:
        params = parse_job_name(job_name)

        script_args = [
            f"--merge_frequency={params['merge_frequency']}",
            f"--prob={params['prob']}",
            f"--init_train_zero={params['init_train_zero']}",
            f"--gaussian_resample={params['gaussian_resample']}",
            f"--deterministic_init={params['deterministic_init']}",
            f"--train_A={params['train_A']}",
            f"--train_B={params['train_B']}",
            f"--init_method_A={params['init_method_A']}",
            f"--init_method_B={params['init_method_B']}",
            f"--use_stochastic={params['use_stochastic']}"
        ]

        # Submit or just create the job script
        submit_slurm_job(
            job_name=job_name,
            script_args=script_args,
            output_dir="slurm_outs",
            sh_folder="slurm_sh",
            partition="batch",
            time="6:00:00",
            mem="12GB",
            gpus=1,
            cpus=16,
            submit=1   # Change to True if you really want to queue them up
        )


[SUBMITTED] Job mf-1_p-0,4_itZ-1_gR-1_det-1_stoch-1_trainA-1_trainB-0_initA-zero_initB-gaussian => Submitted batch job 37119141
[SUBMITTED] Job mf-1_p-0,5_itZ-1_gR-1_det-1_stoch-1_trainA-1_trainB-0_initA-zero_initB-gaussian => Submitted batch job 37119142
[SUBMITTED] Job mf-1_p-0,4_itZ-1_gR-0_det-1_stoch-1_trainA-1_trainB-0_initA-zero_initB-gaussian => Submitted batch job 37119143
[SUBMITTED] Job mf-1_p-0,4_itZ-0_gR-1_det-1_stoch-1_trainA-1_trainB-0_initA-zero_initB-gaussian => Submitted batch job 37119144
[SUBMITTED] Job mf-1_p-0,6_itZ-0_gR-1_det-1_stoch-1_trainA-1_trainB-0_initA-zero_initB-gaussian => Submitted batch job 37119145
[SUBMITTED] Job mf-1_p-0,4_itZ-1_gR-1_det-1_stoch-1_trainA-0_trainB-1_initA-gaussian_initB-zero => Submitted batch job 37119146
[SUBMITTED] Job mf-1_p-0,5_itZ-1_gR-1_det-1_stoch-1_trainA-0_trainB-1_initA-gaussian_initB-zero => Submitted batch job 37119147
[SUBMITTED] Job mf-1_p-0,4_itZ-1_gR-0_det-1_stoch-1_trainA-0_trainB-1_initA-gaussian_initB-zero => Submi

In [ ]:
!tail -n 5 slurm_outs/mf-*.out

In [ ]:
!rm -r slurm_sh/*.sh

In [5]:
%%bash
squeue -u sokoi0a

             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)
          37119141       gpu mf-1_p-0  sokoi0a  R       5:42      1 dgpu501-14
          37119142       gpu mf-1_p-0  sokoi0a  R       5:42      1 dgpu501-14
          37119143       gpu mf-1_p-0  sokoi0a  R       5:42      1 gpu609-03
          37119144       gpu mf-1_p-0  sokoi0a  R       5:42      1 gpu212-10
          37119145       gpu mf-1_p-0  sokoi0a  R       5:42      1 gpu502-01
          37119146       gpu mf-1_p-0  sokoi0a  R       5:42      1 dgpu502-29
          37119147       gpu mf-1_p-0  sokoi0a  R       5:42      1 dgpu501-30
          37119148       gpu mf-1_p-0  sokoi0a  R       5:42      1 dgpu501-26
          37119149       gpu mf-1_p-0  sokoi0a  R       5:42      1 gpu510-12
          37119150       gpu mf-1_p-0  sokoi0a  R       5:42      1 gpu510-07
          37119151       gpu mf-1_p-0  sokoi0a  R       5:42      1 gpu510-02
          37119152       gpu mf-1_p-0  sokoi0a  R   